# Part 2 – Python: Building a Reproducible Traffic Analytics Pipeline

## Task 4 – Mini Traffic Analytics Application


This task develops an interactive command-line application that allows users
to query and analyse the processed traffic dataset.

The application provides three user-focused analytics commands:

1. **Historical Traffic Lookup**  
   Retrieves traffic volume, congestion level, weather conditions, and
   temperature for a specific historical date and time.

2. **Travel Time Optimizer**  
   Allows the user to specify whether they are travelling on a weekday or
   weekend and provide an available travel-time window. The application
   analyses historical hourly traffic patterns within that window and
   recommends the lower-traffic travel time.

3. **Holiday Driving Planner**  
   Allows the user to select a holiday and analyses traffic records from the
   complete calendar dates on which that holiday occurred. Historical hourly
   patterns across multiple years are used to identify relatively quieter and
   busier travel periods.

The application uses an interactive menu and continues running until the user
selects Exit.

Command selections and arguments are logged at INFO level. Invalid user input
is logged at ERROR level and handled with clear user-facing messages rather
than raw Python tracebacks.

### Task 4.1 – Imports and Configuration

The application uses Pandas for querying and analysing the processed traffic
dataset, Path for file handling, and Python's logging module for application
logging.

In [1]:
import logging
from pathlib import Path

import pandas as pd


# ------------------------------------------------------------
# File paths
# ------------------------------------------------------------

PROCESSED_DATA_PATH = Path("engineered_traffic_data.csv")
LOG_PATH = Path("analytics_app.log")


# ------------------------------------------------------------
# Logger
# ------------------------------------------------------------

logger = logging.getLogger(__name__)

### Task 4.2 – Logging Configuration

The application uses a module-level logger created with
`logging.getLogger(__name__)`.

INFO-level messages and above are displayed on the console, while DEBUG-level
messages and above are written to `analytics_app.log`.

The log formatter records the timestamp, logging level, module name and
message.

In [2]:
def configure_logging():
    """Configure console and file logging for the analytics application."""

    logger.setLevel(logging.DEBUG)
    logger.propagate = False

    # Remove existing handlers when re-running the notebook cell.
    for handler in logger.handlers[:]:
        handler.close()
        logger.removeHandler(handler)

    formatter = logging.Formatter(
        "%(asctime)s - %(levelname)s - %(name)s - %(message)s"
    )

    # Console: INFO and above
    console_handler = logging.StreamHandler()
    console_handler.setLevel(logging.INFO)
    console_handler.setFormatter(formatter)

    # File: DEBUG and above
    file_handler = logging.FileHandler(
        LOG_PATH,
        mode="w"
    )
    file_handler.setLevel(logging.DEBUG)
    file_handler.setFormatter(formatter)

    logger.addHandler(console_handler)
    logger.addHandler(file_handler)


configure_logging()

logger.info("Analytics application logging configured successfully.")

2026-10-06 14:03:23,675 - INFO - __main__ - Analytics application logging configured successfully.


### Task 4.3 – Load the Processed Dataset

The application loads the engineered traffic dataset produced in Task 2.

The date/time column is parsed as a Pandas datetime variable and the required
columns for the three analytics commands are validated before the application
continues.

In [3]:
def load_processed_data(file_path):
    """Load and validate the processed traffic dataset."""

    required_columns = {
        "date_time",
        "traffic_volume",
        "weather_main",
        "weather_description",
        "temp",
        "congestion_category",
        "hour",
        "is_weekend",
        "holiday"
    }

    try:
        df = pd.read_csv(
            file_path,
            parse_dates=["date_time"]
        )

        missing_columns = required_columns - set(df.columns)

        if missing_columns:
            logger.error(
                "Processed dataset is missing required columns: %s",
                sorted(missing_columns)
            )
            return None

        logger.info(
            "Processed dataset loaded successfully: %d rows, %d columns.",
            df.shape[0],
            df.shape[1]
        )

        return df

    except FileNotFoundError:
        logger.error(
            "Processed dataset was not found: %s",
            file_path
        )
        return None

    except (pd.errors.ParserError, ValueError) as error:
        logger.error(
            "Unable to parse processed dataset: %s",
            error
        )
        return None

    except OSError as error:
        logger.error(
            "File I/O error while loading processed dataset: %s",
            error
        )
        return None


traffic_df = load_processed_data(PROCESSED_DATA_PATH)

traffic_df.shape if traffic_df is not None else None



2026-10-06 14:03:24,013 - INFO - __main__ - Processed dataset loaded successfully: 48187 rows, 30 columns.


(48187, 30)

### Task 4.4 – Helper Functions

Some timestamps can contain more than one weather observation. For traffic
pattern analysis, the application creates one traffic observation per unique
timestamp so that timestamps with multiple weather descriptions do not receive
additional weight.

A helper function is also used to validate user-entered hourly travel times.

In [4]:
def build_unique_traffic_records(df):
    """Create one traffic-volume observation for each unique timestamp."""

    records = (
        df
        .groupby("date_time", as_index=False)
        .agg(
            traffic_volume=("traffic_volume", "mean"),
            hour=("hour", "first"),
            is_weekend=("is_weekend", "first")
        )
    )

    records["date"] = records["date_time"].dt.date
    records["year"] = records["date_time"].dt.year

    return records


def parse_hour_input(value):
    """
    Convert user-entered hour values such as '6', '06' or '06:00'
    into an integer from 0 to 23.
    """

    text = str(value).strip()

    if ":" in text:
        parsed_time = pd.to_datetime(
            text,
            format="%H:%M",
            errors="raise"
        )

        if parsed_time.minute != 0:
            raise ValueError(
                "Only whole-hour values are supported."
            )

        hour = parsed_time.hour

    else:
        hour = int(text)

    if hour < 0 or hour > 23:
        raise ValueError(
            "Hour must be between 0 and 23."
        )

    return hour


def format_hour(hour):
    """Format an integer hour as HH:00."""

    return f"{int(hour):02d}:00"

### Task 4.5 – Command 1: Historical Traffic Lookup

This command allows the user to investigate a specific historical date and
hour.

Because the dataset contains hourly traffic observations, the user enters:

- A date in `YYYY-MM-DD` format
- An hour as a whole number from `0` to `23`

The result includes:

- Traffic volume
- Congestion category
- Weather condition
- Weather description
- Temperature

Invalid dates or hours are logged at ERROR level and returned to the user as
clear messages without displaying raw Python tracebacks.

In [5]:
def query_historical_traffic(df, date_text, hour_text):
    """Retrieve traffic and weather information for a specific date and hour."""

    logger.info(
        (
            "Command invoked: historical-lookup, "
            "arguments: date=%s, hour=%s"
        ),
        date_text,
        hour_text
    )

    # --------------------------------------------------------
    # Validate date
    # --------------------------------------------------------

    try:
        date_text = str(date_text).strip()

        # Require exact YYYY-MM-DD format.
        if (
            len(date_text) != 10
            or date_text[4] != "-"
            or date_text[7] != "-"
        ):
            raise ValueError

        target_date = pd.to_datetime(
            date_text,
            format="%Y-%m-%d",
            errors="raise"
        )

        # Confirm that the parsed date exactly matches the input.
        if target_date.strftime("%Y-%m-%d") != date_text:
            raise ValueError

    except (ValueError, TypeError):
        logger.error(
            "Invalid date input: %s",
            date_text
        )

        return {
            "success": False,
            "message": (
                "Invalid date. "
                "Please use YYYY-MM-DD format."
            )
        }

    # --------------------------------------------------------
    # Validate hour
    # --------------------------------------------------------

    try:
        hour_text = str(hour_text).strip()

        # Whole-number hours only.
        if not hour_text.isdigit():
            raise ValueError

        target_hour = int(hour_text)

        if target_hour < 0 or target_hour > 23:
            raise ValueError

    except (ValueError, TypeError):
        logger.error(
            "Invalid hour input: %s",
            hour_text
        )

        return {
            "success": False,
            "message": (
                "Invalid hour. "
                "Please enter a whole number from 0 to 23."
            )
        }

    # --------------------------------------------------------
    # Check whether the selected date exists
    # --------------------------------------------------------

    selected_date_records = df.loc[
        df["date_time"].dt.date
        == target_date.date()
    ]

    if selected_date_records.empty:
        logger.warning(
            "No traffic data available for date=%s",
            date_text
        )

        return {
            "success": False,
            "message": (
                "No traffic data is available for the selected date. "
                "Please choose another date."
            )
        }

    # --------------------------------------------------------
    # Build requested hourly timestamp
    # --------------------------------------------------------

    target_datetime = (
        target_date
        + pd.Timedelta(hours=target_hour)
    )

    columns = [
        "date_time",
        "traffic_volume",
        "congestion_category",
        "weather_main",
        "weather_description",
        "temp"
    ]

    result = (
        df.loc[
            df["date_time"] == target_datetime,
            columns
        ]
        .drop_duplicates()
        .reset_index(drop=True)
    )

    # --------------------------------------------------------
    # Date exists, but selected hour does not
    # --------------------------------------------------------

    if result.empty:
        logger.warning(
            (
                "No traffic data available for "
                "date=%s, hour=%d"
            ),
            date_text,
            target_hour
        )

        return {
            "success": False,
            "message": (
                "Traffic data is available for this date, "
                "but not for the selected hour. "
                "Please choose another hour."
            )
        }

    # --------------------------------------------------------
    # Successful lookup
    # --------------------------------------------------------

    logger.info(
        "Historical lookup returned %d matching record(s).",
        len(result)
    )

    return {
        "success": True,
        "data": result
    }

In [6]:
test_lookup = query_historical_traffic(
    traffic_df,
    "2016-01-19",
    "9"
)

test_lookup

2026-10-06 14:03:24,077 - INFO - __main__ - Command invoked: historical-lookup, arguments: date=2016-01-19, hour=9
2026-10-06 14:03:24,137 - WARNING - __main__ - No traffic data available for date=2016-01-19, hour=9


{'success': False,
 'message': 'Traffic data is available for this date, but not for the selected hour. Please choose another hour.'}

### Task 4.6 – Command 2: Travel Time Optimizer

The Travel Time Optimizer helps a user decide when to travel within a time
window that is practical for them.

The user specifies:

- Weekday or weekend travel
- Earliest acceptable travel hour
- Latest acceptable travel hour

The application analyses historical traffic records within that constraint and
calculates:

- The historically lower-traffic hour
- The historically busiest hour
- Average traffic at both times
- The difference in average traffic volume
- The percentage reduction associated with choosing the lower-traffic hour

The recommendation represents a historical traffic pattern rather than a
real-time traffic forecast.



In [7]:
def optimize_travel_time(
    df,
    day_type,
    start_hour,
    end_hour
):
    """
    Find the historically lower-traffic hour within a user-defined
    weekday/weekend travel window.
    """

    logger.info(
        (
            "Command invoked: travel-time-optimizer, "
            "arguments: day_type=%s, start_hour=%s, end_hour=%s"
        ),
        day_type,
        start_hour,
        end_hour
    )

    day_type_clean = str(day_type).strip().lower()

    if day_type_clean not in {"weekday", "weekend"}:
        logger.error(
            "Invalid day type: %s",
            day_type
        )

        return {
            "success": False,
            "message": (
                "Invalid day type. "
                "Please select Weekday or Weekend."
            )
        }

    try:
        start = parse_hour_input(start_hour)
        end = parse_hour_input(end_hour)

    except (ValueError, TypeError) as error:
        logger.error(
            "Invalid travel-time input: %s",
            error
        )

        return {
            "success": False,
            "message": (
                "Invalid travel time. "
                "Please enter a whole hour between 00:00 and 23:00."
            )
        }

    if start > end:
        logger.error(
            "Invalid travel window: start=%s, end=%s",
            start_hour,
            end_hour
        )

        return {
            "success": False,
            "message": (
                "The earliest travel time must be before "
                "or equal to the latest travel time."
            )
        }

    records = build_unique_traffic_records(df)

    weekend_flag = (
        1 if day_type_clean == "weekend" else 0
    )

    selected_records = records.loc[
        (records["is_weekend"] == weekend_flag)
        & (records["hour"] >= start)
        & (records["hour"] <= end)
    ].copy()

    if selected_records.empty:
        logger.warning(
            (
                "No historical records found for "
                "day_type=%s, start=%d, end=%d"
            ),
            day_type_clean,
            start,
            end
        )

        return {
            "success": False,
            "message": (
                "No historical traffic records were found "
                "for that travel window."
            )
        }

    hourly_summary = (
        selected_records
        .groupby("hour")["traffic_volume"]
        .agg(
            average_traffic="mean",
            median_traffic="median",
            observations="count"
        )
        .sort_index()
    )

    logger.debug(
        "Travel optimizer hourly summary: %s",
        hourly_summary.to_dict("index")
    )

    best_hour = int(
        hourly_summary["average_traffic"].idxmin()
    )

    busiest_hour = int(
        hourly_summary["average_traffic"].idxmax()
    )

    best_average = float(
        hourly_summary.loc[
            best_hour,
            "average_traffic"
        ]
    )

    busiest_average = float(
        hourly_summary.loc[
            busiest_hour,
            "average_traffic"
        ]
    )

    traffic_difference = (
        busiest_average - best_average
    )

    if busiest_average > 0:
        reduction_percentage = (
            traffic_difference
            / busiest_average
            * 100
        )
    else:
        reduction_percentage = 0.0

    logger.info(
        (
            "Travel optimizer completed: "
            "recommended_hour=%s, busiest_hour=%s"
        ),
        format_hour(best_hour),
        format_hour(busiest_hour)
    )

    return {
        "success": True,
        "day_type": day_type_clean.title(),
        "start_hour": start,
        "end_hour": end,
        "best_hour": best_hour,
        "best_average": best_average,
        "busiest_hour": busiest_hour,
        "busiest_average": busiest_average,
        "traffic_difference": traffic_difference,
        "reduction_percentage": reduction_percentage,
        "observations": len(selected_records),
        "hourly_summary": hourly_summary
    }

In [8]:
test_optimizer = optimize_travel_time(
    traffic_df,
    "weekday",
    "06:00",
    "10:00"
)

test_optimizer

2026-10-06 14:03:24,173 - INFO - __main__ - Command invoked: travel-time-optimizer, arguments: day_type=weekday, start_hour=06:00, end_hour=10:00
2026-10-06 14:03:24,256 - INFO - __main__ - Travel optimizer completed: recommended_hour=10:00, busiest_hour=07:00


{'success': True,
 'day_type': 'Weekday',
 'start_hour': 6,
 'end_hour': 10,
 'best_hour': 10,
 'best_average': 4400.511513157895,
 'busiest_hour': 7,
 'busiest_average': 6061.162457912458,
 'traffic_difference': 1660.6509447545632,
 'reduction_percentage': 27.3982252791573,
 'observations': 5982,
 'hourly_summary':       average_traffic  median_traffic  observations
 hour                                               
 6         5375.273780          5588.0          1209
 7         6061.162458          6320.5          1188
 8         5508.896580          5751.0          1199
 9         4900.601709          5053.0          1170
 10        4400.511513          4484.5          1216}

### Task 4.7 – Command 3: Holiday Driving Planner

The Holiday Driving Planner helps users make travel decisions around major
holidays using historical traffic patterns.

The holiday field may identify only a single record on each holiday date.
Therefore, the application first identifies the calendar dates associated
with the selected holiday and then retrieves all available traffic records
from those complete dates.

The analysis focuses on practical driving hours from 06:00 to 22:00 and
combines multiple historical occurrences of the selected holiday.

The application calculates:

- Historical holiday dates available
- Average traffic by hour
- Average traffic during morning, afternoon and evening periods
- Historically lower-traffic and busier periods
- Lower-traffic and busiest individual hours
- Year-by-year average traffic patterns

The results represent historical planning guidance rather than a prediction
of future holiday traffic.


In [9]:
def get_available_holidays(df):
    """Return valid holiday names available in the dataset."""

    holiday_values = (
        df["holiday"]
        .dropna()
        .astype(str)
        .str.strip()
    )

    holiday_values = holiday_values.loc[
        (holiday_values != "")
        & (holiday_values.str.lower() != "none")
    ]

    return sorted(
        holiday_values.unique().tolist()
    )


def holiday_driving_plan(df, holiday_name):
    """
    Analyse complete historical dates associated with a selected holiday.
    """

    logger.info(
        "Command invoked: holiday-driving-planner, arguments: holiday=%s",
        holiday_name
    )

    available_holidays = get_available_holidays(df)

    holiday_lookup = {
        holiday.lower(): holiday
        for holiday in available_holidays
    }

    requested_holiday = str(
        holiday_name
    ).strip().lower()

    # --------------------------------------------------------
    # Validate holiday selection
    # --------------------------------------------------------

    if requested_holiday not in holiday_lookup:

        logger.error(
            "Invalid holiday selection: %s",
            holiday_name
        )

        return {
            "success": False,
            "message": (
                "Invalid holiday. "
                "Please select one of the available holidays."
            )
        }

    official_holiday_name = holiday_lookup[
        requested_holiday
    ]

    # --------------------------------------------------------
    # Identify historical holiday dates
    # --------------------------------------------------------

    holiday_rows = df.loc[
        df["holiday"]
        .fillna("")
        .astype(str)
        .str.strip()
        .str.lower()
        == official_holiday_name.lower()
    ].copy()

    holiday_dates = sorted(
        holiday_rows["date_time"]
        .dt.date
        .unique()
        .tolist()
    )

    if not holiday_dates:

        logger.error(
            "No historical dates found for holiday=%s",
            official_holiday_name
        )

        return {
            "success": False,
            "message": (
                "No historical dates were found "
                "for the selected holiday."
            )
        }

    logger.debug(
        "Historical dates for %s: %s",
        official_holiday_name,
        holiday_dates
    )

    # --------------------------------------------------------
    # Retrieve complete traffic records for holiday dates
    # --------------------------------------------------------

    records = build_unique_traffic_records(df)

    holiday_records = records.loc[
        records["date"].isin(holiday_dates)
    ].copy()

    # Practical driving hours: 06:00–22:00
    practical_records = holiday_records.loc[
        holiday_records["hour"].between(
            6,
            22
        )
    ].copy()

    if practical_records.empty:

        logger.warning(
            "No practical-hour traffic records found for %s",
            official_holiday_name
        )

        return {
            "success": False,
            "message": (
                "No traffic records between 06:00 and 22:00 "
                "were found for the selected holiday."
            )
        }

    # --------------------------------------------------------
    # Hourly traffic analysis
    # --------------------------------------------------------

    hourly_summary = (
        practical_records
        .groupby("hour")["traffic_volume"]
        .agg(
            average_traffic="mean",
            observations="count"
        )
        .sort_index()
    )

    best_hour = int(
        hourly_summary["average_traffic"].idxmin()
    )

    busiest_hour = int(
        hourly_summary["average_traffic"].idxmax()
    )

    best_hour_average = float(
        hourly_summary.loc[
            best_hour,
            "average_traffic"
        ]
    )

    busiest_hour_average = float(
        hourly_summary.loc[
            busiest_hour,
            "average_traffic"
        ]
    )

    # --------------------------------------------------------
    # Morning / afternoon / evening analysis
    # --------------------------------------------------------

    def assign_travel_period(hour):

        if 6 <= hour <= 11:
            return "Morning (06:00-11:00)"

        elif 12 <= hour <= 17:
            return "Afternoon (12:00-17:00)"

        else:
            return "Evening (18:00-22:00)"

    practical_records["travel_period"] = (
        practical_records["hour"]
        .apply(assign_travel_period)
    )

    period_order = [
        "Morning (06:00-11:00)",
        "Afternoon (12:00-17:00)",
        "Evening (18:00-22:00)"
    ]

    period_summary = (
        practical_records
        .groupby("travel_period")["traffic_volume"]
        .mean()
        .reindex(period_order)
        .dropna()
    )

    best_period = str(
        period_summary.idxmin()
    )

    busiest_period = str(
        period_summary.idxmax()
    )

    best_period_average = float(
        period_summary.loc[best_period]
    )

    busiest_period_average = float(
        period_summary.loc[busiest_period]
    )

    # --------------------------------------------------------
    # Year-by-year holiday trend
    # --------------------------------------------------------

    yearly_summary = (
        practical_records
        .groupby("year")["traffic_volume"]
        .mean()
        .sort_index()
    )

    if len(yearly_summary) >= 2:

        first_year = int(
            yearly_summary.index[0]
        )

        last_year = int(
            yearly_summary.index[-1]
        )

        first_average = float(
            yearly_summary.iloc[0]
        )

        last_average = float(
            yearly_summary.iloc[-1]
        )

        if first_average > 0:

            trend_percentage = (
                (last_average - first_average)
                / first_average
                * 100
            )

        else:
            trend_percentage = 0.0

    else:

        first_year = int(
            yearly_summary.index[0]
        )

        last_year = first_year
        trend_percentage = None

    logger.debug(
        "Holiday hourly summary for %s: %s",
        official_holiday_name,
        hourly_summary.to_dict("index")
    )

    logger.debug(
        "Holiday period summary for %s: %s",
        official_holiday_name,
        period_summary.to_dict()
    )

    logger.debug(
        "Holiday yearly summary for %s: %s",
        official_holiday_name,
        yearly_summary.to_dict()
    )

    logger.info(
        (
            "Holiday driving analysis completed for %s "
            "using %d historical holiday date(s)."
        ),
        official_holiday_name,
        len(holiday_dates)
    )

    return {
        "success": True,
        "holiday": official_holiday_name,
        "holiday_dates": holiday_dates,
        "historical_dates": len(holiday_dates),
        "observations": len(practical_records),
        "best_hour": best_hour,
        "best_hour_average": best_hour_average,
        "busiest_hour": busiest_hour,
        "busiest_hour_average": busiest_hour_average,
        "best_period": best_period,
        "best_period_average": best_period_average,
        "busiest_period": busiest_period,
        "busiest_period_average": busiest_period_average,
        "period_summary": period_summary,
        "yearly_summary": yearly_summary,
        "first_year": first_year,
        "last_year": last_year,
        "trend_percentage": trend_percentage,
        "hourly_summary": hourly_summary
    }

In [10]:
test_holiday = holiday_driving_plan(
    traffic_df,
    "Labor Day"
)

test_holiday

2026-10-06 14:03:24,304 - INFO - __main__ - Command invoked: holiday-driving-planner, arguments: holiday=Labor Day
2026-10-06 14:03:24,397 - INFO - __main__ - Holiday driving analysis completed for Labor Day using 5 historical holiday date(s).


{'success': True,
 'holiday': 'Labor Day',
 'holiday_dates': [datetime.date(2013, 9, 2),
  datetime.date(2015, 9, 7),
  datetime.date(2016, 9, 5),
  datetime.date(2017, 9, 4),
  datetime.date(2018, 9, 3)],
 'historical_dates': 5,
 'observations': 84,
 'best_hour': 6,
 'best_hour_average': 1090.8,
 'busiest_hour': 14,
 'busiest_hour_average': 4400.6,
 'best_period': 'Morning (06:00-11:00)',
 'best_period_average': 2307.3793103448274,
 'busiest_period': 'Afternoon (12:00-17:00)',
 'busiest_period_average': 4127.666666666667,
 'period_summary': travel_period
 Morning (06:00-11:00)      2307.379310
 Afternoon (12:00-17:00)    4127.666667
 Evening (18:00-22:00)      2625.600000
 Name: traffic_volume, dtype: float64,
 'yearly_summary': year
 2013    3109.375000
 2015    3199.529412
 2016    2798.941176
 2017    3129.647059
 2018    3026.823529
 Name: traffic_volume, dtype: float64,
 'first_year': 2013,
 'last_year': 2018,
 'trend_percentage': -2.654921667159321,
 'hourly_summary':       aver

### Task 4.8 – User-Facing Result Display

The following functions format the results of the three analytics commands for
the command-line user.

`print()` is used only for direct output intended for the user of the CLI
application. Internal application events, command arguments and errors are
handled using Python logging.

In [11]:
def display_historical_result(result):
    """Display Historical Traffic Lookup results."""

    if not result["success"]:
        print(f"\n{result['message']}")
        return

    print("\n" + "=" * 60)
    print("HISTORICAL TRAFFIC LOOKUP")
    print("=" * 60)

    print(
        result["data"].to_string(
            index=False
        )
    )


def display_optimizer_result(result):
    """Display Travel Time Optimizer results."""

    if not result["success"]:
        print(f"\n{result['message']}")
        return

    print("\n" + "=" * 60)
    print("TRAVEL TIME OPTIMIZER")
    print("=" * 60)

    print(f"Day type: {result['day_type']}")

    print(
        f"Travel window: "
        f"{format_hour(result['start_hour'])}"
        f"-{format_hour(result['end_hour'])}"
    )

    print(
        f"Historical observations: "
        f"{result['observations']:,}"
    )

    print("\nRecommended lower-traffic time:")

    print(
        f"  {format_hour(result['best_hour'])}"
        f" | Average traffic: "
        f"{result['best_average']:,.0f}"
    )

    print("\nBusiest time in this window:")

    print(
        f"  {format_hour(result['busiest_hour'])}"
        f" | Average traffic: "
        f"{result['busiest_average']:,.0f}"
    )

    print(
        f"\nHistorical traffic difference: "
        f"{result['traffic_difference']:,.0f} vehicles"
    )

    print(
        f"Potential reduction: "
        f"{result['reduction_percentage']:.1f}%"
    )

    print(
        "\nNote: This recommendation is based on historical "
        "traffic patterns, not a real-time forecast."
    )


def display_holiday_result(result):
    """Display Holiday Driving Planner results."""

    if not result["success"]:
        print(f"\n{result['message']}")
        return

    print("\n" + "=" * 60)
    print("HOLIDAY DRIVING PLANNER")
    print("=" * 60)

    print(
        f"Holiday: {result['holiday']}"
    )

    print(
        f"Historical dates used: "
        f"{result['historical_dates']}"
    )

    print(
        "Years analysed: "
        + ", ".join(
            str(date.year)
            for date in result["holiday_dates"]
        )
    )

    print(
        f"Traffic observations: "
        f"{result['observations']:,}"
    )

    print(
        "Driving hours analysed: 06:00-22:00"
    )

    print("\nRecommended historical travel period:")

    print(
        f"  {result['best_period']}"
        f" | Average traffic: "
        f"{result['best_period_average']:,.0f}"
    )

    print("\nHistorically busiest period:")

    print(
        f"  {result['busiest_period']}"
        f" | Average traffic: "
        f"{result['busiest_period_average']:,.0f}"
    )

    print("\nLower-traffic historical hour:")

    print(
        f"  {format_hour(result['best_hour'])}"
        f" | Average traffic: "
        f"{result['best_hour_average']:,.0f}"
    )

    print("\nHistorically busiest hour:")

    print(
        f"  {format_hour(result['busiest_hour'])}"
        f" | Average traffic: "
        f"{result['busiest_hour_average']:,.0f}"
    )

    print(
        "\nNote: This is historical planning guidance "
        "based on available records, not a forecast."
    )

### Task 4.9 – Interactive Command-Line Menu

The application provides an interactive menu with three analytics commands.

1. Historical Traffic Lookup
2. Travel Time Optimizer
3. Holiday Driving Planner
4. Exit

The menu requests only the arguments required by the selected command and
continues running until the user selects Exit.

Command names and arguments are recorded at INFO level. Invalid user input is
logged at ERROR level and handled with a clear message instead of a raw Python
traceback.

In [12]:
def main_menu(df):
    """Run the interactive Smart City Traffic Analytics application."""

    while True:

        print("\n" + "=" * 60)
        print("             SMART CITY TRAFFIC ANALYTICS")
        print("=" * 60)

        print("\n1. Historical Traffic Lookup")
        print(
            "   Check traffic, congestion and weather "
            "for a specific historical date and hour."
        )

        print("\n2. Travel Time Optimizer")
        print(
            "   Find a lower-traffic hour within "
            "your available travel window."
        )

        print("\n3. Holiday Driving Planner")
        print(
            "   Use historical holiday traffic to "
            "identify better driving periods."
        )

        print("\n4. Exit")

        choice = input(
            "\nSelect a menu number (1-4): "
        ).strip()

        # ====================================================
        # Command 1 – Historical Traffic Lookup
        # ====================================================

        if choice == "1":

            date_text = input(
                "\nEnter date (YYYY-MM-DD): "
            ).strip()

            hour_text = input(
                "Enter hour (0-23): "
            ).strip()

            result = query_historical_traffic(
                df,
                date_text,
                hour_text
            )

            display_historical_result(result)

        # ====================================================
        # Command 2 – Travel Time Optimizer
        # ====================================================

        elif choice == "2":

            print("\nSelect day type:")
            print("1. Weekday")
            print("2. Weekend")

            day_choice = input(
                "Select 1 or 2: "
            ).strip()

            if day_choice == "1":
                day_type = "weekday"

            elif day_choice == "2":
                day_type = "weekend"

            else:
                logger.info(
                    (
                        "Command invoked: travel-time-optimizer, "
                        "arguments: day_selection=%s"
                    ),
                    day_choice
                )

                logger.error(
                    "Invalid day-type selection: %s",
                    day_choice
                )

                print(
                    "\nInvalid selection. "
                    "Please select 1 or 2."
                )

                continue

            start_hour = input(
                "\nEarliest travel hour (0-23): "
            ).strip()

            end_hour = input(
                "Latest travel hour (0-23): "
            ).strip()

            result = optimize_travel_time(
                df,
                day_type,
                start_hour,
                end_hour
            )

            display_optimizer_result(result)

        # ====================================================
        # Command 3 – Holiday Driving Planner
        # ====================================================

        elif choice == "3":

            holidays = get_available_holidays(df)

            print("\nAvailable holidays:")

            for number, holiday in enumerate(
                holidays,
                start=1
            ):
                print(
                    f"{number}. {holiday}"
                )

            selection = input(
                "\nSelect a holiday number: "
            ).strip()

            try:
                holiday_number = int(selection)

                if not 1 <= holiday_number <= len(holidays):
                    raise ValueError

            except ValueError:

                logger.info(
                    (
                        "Command invoked: holiday-driving-planner, "
                        "arguments: selection=%s"
                    ),
                    selection
                )

                logger.error(
                    "Invalid holiday selection: %s",
                    selection
                )

                print(
                    "\nInvalid holiday selection. "
                    "Please choose a number from the list."
                )

                continue

            selected_holiday = holidays[
                holiday_number - 1
            ]

            result = holiday_driving_plan(
                df,
                selected_holiday
            )

            display_holiday_result(result)

        # ====================================================
        # Exit
        # ====================================================

        elif choice == "4":

            logger.info(
                "Command invoked: exit, arguments: none"
            )

            print(
                "\nExiting Smart City Traffic Analytics."
            )

            break

        # ====================================================
        # Invalid menu selection
        # ====================================================

        else:

            logger.error(
                "Invalid menu selection: %s",
                choice
            )

            print(
                "\nInvalid choice. "
                "Please enter a number from 1 to 4."
            )

### Task 4.10 – Validation

The three analytics commands are tested without launching the interactive menu
so that the notebook can run from beginning to end without waiting for user
input.

Invalid input is also tested to confirm that the application returns clear
error messages without generating raw Python tracebacks.

In [13]:
# ------------------------------------------------------------
# Valid command tests
# ------------------------------------------------------------

test_lookup = query_historical_traffic(
    traffic_df,
    "2016-10-02",
    "9"
)

test_optimizer = optimize_travel_time(
    traffic_df,
    "weekday",
    "6",
    "10"
)

test_holiday = holiday_driving_plan(
    traffic_df,
    "Labor Day"
)


valid_tests = {
    "Historical Traffic Lookup":
        test_lookup["success"],

    "Travel Time Optimizer":
        test_optimizer["success"],

    "Holiday Driving Planner":
        test_holiday["success"]
}


# ------------------------------------------------------------
# Invalid input tests
# ------------------------------------------------------------

invalid_date = query_historical_traffic(
    traffic_df,
    "2016-10-02",
    "9"
)

invalid_hour = query_historical_traffic(
    traffic_df,
    "2016-01-19",
    "25"
)

invalid_optimizer = optimize_travel_time(
    traffic_df,
    "weekday",
    "25",
    "10"
)

invalid_holiday = holiday_driving_plan(
    traffic_df,
    "Not A Real Holiday"
)


invalid_tests = {
    "Malformed date handled":
        not invalid_date["success"],

    "Invalid hour handled":
        not invalid_hour["success"],

    "Invalid optimizer input handled":
        not invalid_optimizer["success"],

    "Invalid holiday handled":
        not invalid_holiday["success"]
}


print("Valid command tests:")
print(valid_tests)

print("\nInvalid input tests:")
print(invalid_tests)

2026-10-06 14:03:24,481 - INFO - __main__ - Command invoked: historical-lookup, arguments: date=2016-10-02, hour=9
2026-10-06 14:03:24,515 - INFO - __main__ - Historical lookup returned 1 matching record(s).
2026-10-06 14:03:24,517 - INFO - __main__ - Command invoked: travel-time-optimizer, arguments: day_type=weekday, start_hour=6, end_hour=10
2026-10-06 14:03:24,564 - INFO - __main__ - Travel optimizer completed: recommended_hour=10:00, busiest_hour=07:00
2026-10-06 14:03:24,567 - INFO - __main__ - Command invoked: holiday-driving-planner, arguments: holiday=Labor Day
2026-10-06 14:03:24,654 - INFO - __main__ - Holiday driving analysis completed for Labor Day using 5 historical holiday date(s).
2026-10-06 14:03:24,656 - INFO - __main__ - Command invoked: historical-lookup, arguments: date=2016-10-02, hour=9
2026-10-06 14:03:24,688 - INFO - __main__ - Historical lookup returned 1 matching record(s).
2026-10-06 14:03:24,690 - INFO - __main__ - Command invoked: historical-lookup, argume

Valid command tests:
{'Historical Traffic Lookup': True, 'Travel Time Optimizer': True, 'Holiday Driving Planner': True}

Invalid input tests:
{'Malformed date handled': False, 'Invalid hour handled': True, 'Invalid optimizer input handled': True, 'Invalid holiday handled': True}


### Task 4.11 – Task 4 Summary

The mini traffic analytics application provides three distinct user queries:

1. **Historical Traffic Lookup** retrieves traffic, congestion and weather
   information for a selected historical date and hour.

2. **Travel Time Optimizer** analyses historical weekday or weekend traffic
   within a user-defined travel window and identifies a historically
   lower-traffic travel hour.

3. **Holiday Driving Planner** analyses complete historical holiday dates and
   identifies relatively quieter and busier driving periods.

The application records command names and arguments at INFO level and records
invalid user input at ERROR level.

Invalid dates, hours and menu selections are handled with clear user-facing
messages rather than raw Python tracebacks.

The interactive application uses `print()` only for output intended directly
for the CLI user.

In [14]:
 ## main_menu(traffic_df)